## 現状の問題

各 `+`, `*` のたびに `Value` オブジェクト（+ 2つの `Vector` ヒープ確保）が生成されます。`linear(x, w)` で x が長さ16、w が 16x16 なら、1回の呼び出しで ~500個の `Value` ノードが生成されます。シーケンス長5の forward pass 全体では**数十万ノード**になり、ほぼ全ての時間がメモリ確保に費やされています。

---

## 案1: Value のアロケーション削減（効果: 中、難易度: 低）

`_children` と `_local_grads` が毎回 `Vector`（ヒープ確保）になっている。演算の子ノードは最大2個なので、固定サイズにできる：

```julia
mutable struct Value
    data::Float64
    grad::Float64
    child1::Union{Value, Nothing}
    child2::Union{Value, Nothing}
    lg1::Float64
    lg2::Float64
end
```

Vector 2本のヒープ確保がなくなり、ノードあたりのコストが大幅に減る。

---

## 案2: 行列レベルの autograd（効果: 大、難易度: 中）

現在はスカラー1個ずつの計算グラフ。`linear(x, w)` を1つの autograd ノードにすれば、ノード数が **数桁** 減る：

```julia
# 現状: 16x16 = 256回の *, 240回の + → ~500ノード
# 改善後: linear 1回 → 1ノード（中身は普通の行列積）
function linear_fwd(x::Vector{Float64}, w::Matrix{Float64})
    w * x  # BLAS で高速
end
# backward で ∂L/∂x = wᵀ g, ∂L/∂w = g xᵀ を一括計算
```

`softmax`, `rmsnorm` も同様にまとめられる。これが**最も効果が大きい**。

---

## 案3: テープベース autograd（効果: 大、難易度: 中）

各 `Value` にグラフを持たせる代わりに、グローバルな「テープ」に演算を記録する：

```julia
struct TapeEntry
    op::Symbol        # :add, :mul, :matmul, ...
    result_idx::Int
    input_idxs::NTuple{2,Int}
    cached::NTuple{2,Float64}  # backward に必要な値
end

tape = TapeEntry[]  # 1本の Vector に全演算を記録
```

メモリが連続的でキャッシュフレンドリー。backward はテープを逆順に辿るだけ。PyTorch もこの方式。

---

## 案4: `@inbounds` + 型安定化（効果: 小〜中、難易度: 低）

ホットループに `@inbounds` を付け、型不安定を除去：

```julia
function linear(x::Vector{Value}, w::Vector{Vector{Value}})
    @inbounds [sum(wi * xi for (wi, xi) in zip(wo, x)) for wo in w]
end
```

また `sum(generator)` は型推論が弱い場合がある。明示的なループに展開すると改善する場合がある。

---

## 案5: 既存 AD フレームワークの利用（効果: 特大、難易度: 低）

自作 autograd をやめて Zygote.jl や Enzyme.jl を使う：

```julia
using Zygote
grads = gradient(params) do
    loss = forward(params, data)
end
```

コンパイラレベルで最適化され、BLAS も活用される。ただし「教育用の最小実装」という趣旨からは外れる。

---

## まとめ

| 案 | 効果 | 難易度 | 趣旨を維持 |
|---|---|---|---|
| 1. Value 構造体のスリム化 | 中 | 低 | はい |
| 2. 行列レベル autograd | **大** | 中 | はい |
| 3. テープベース autograd | **大** | 中 | はい |
| 4. @inbounds + 型安定化 | 小〜中 | 低 | はい |
| 5. Zygote/Enzyme | 特大 | 低 | いいえ |

教育目的を維持しつつ最も効果が高いのは**案2（行列レベル autograd）**です。スカラー演算をまとめるだけでノード数が数百分の一になり、さらに BLAS の恩恵も受けられます。
    